# 03. Tool Calling 평가

이 노트북에서는 Fine-tuning된 모델의 Tool Calling 정확도를 평가합니다.

## 목차
1. 환경 설정
2. 모델 로드
3. 테스트 데이터 준비
4. Tool Calling 평가
5. 에러 분석
6. 결과 저장

## 1. 환경 설정

In [ ]:
import os
import sys
sys.path.append('..')

import torch
import json
from tqdm import tqdm
from dotenv import load_dotenv

load_dotenv('../.env')

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
import yaml

from src.evaluation import (
    extract_tool_call,
    compare_tool_calls,
    evaluate_model,
    calculate_metrics,
)
from src.data_utils import AVAILABLE_TOOLS, SYSTEM_PROMPT

print("라이브러리 로드 완료!")

## 2. 모델 로드

평가할 실험을 선택합니다.

In [ ]:
# 평가할 실험 선택
EXPERIMENT = "exp-001"  # exp-001, exp-002, exp-003 중 선택

# 설정 로드
config_path = f'../configs/training_config_{EXPERIMENT.replace("-", "")}.yaml'
with open(config_path, 'r', encoding='utf-8') as f:
    config = yaml.safe_load(f)

print(f"평가 실험: {config['experiment']['name']}")
print(f"설명: {config['experiment']['description']}")

In [ ]:
# 체크포인트 경로
checkpoint_dir = f"../{config['misc']['output_dir']}/final_checkpoint"

if not os.path.exists(checkpoint_dir):
    print(f"경고: 체크포인트가 없습니다: {checkpoint_dir}")
    print("먼저 02_qlora_training.ipynb를 실행하세요.")
else:
    print(f"체크포인트 발견: {checkpoint_dir}")

In [ ]:
# 4-bit 양자화 설정
model_config = config['model']

bnb_config = BitsAndBytesConfig(
    load_in_4bit=model_config['load_in_4bit'],
    bnb_4bit_quant_type=model_config['bnb_4bit_quant_type'],
    bnb_4bit_compute_dtype=getattr(torch, model_config['bnb_4bit_compute_dtype']),
    bnb_4bit_use_double_quant=model_config['bnb_4bit_use_double_quant'],
)

print("BitsAndBytes 설정 완료")

In [ ]:
# Base 모델 로드
print(f"Base 모델 로딩 중: {model_config['base_model']}")

base_model = AutoModelForCausalLM.from_pretrained(
    model_config['base_model'],
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=model_config['trust_remote_code'],
)

print("Base 모델 로드 완료!")

In [ ]:
# LoRA 어댑터 로드
print(f"LoRA 어댑터 로딩 중: {checkpoint_dir}")

model = PeftModel.from_pretrained(
    base_model,
    checkpoint_dir,
)
model.eval()

print("LoRA 어댑터 로드 완료!")

In [ ]:
# 토크나이저 로드
tokenizer = AutoTokenizer.from_pretrained(
    checkpoint_dir,
    trust_remote_code=model_config['trust_remote_code'],
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("토크나이저 로드 완료!")

## 3. 테스트 데이터 준비

In [ ]:
# 테스트 케이스 로드
test_cases_path = '../data/evaluation/tool_calling_test_cases.json'

with open(test_cases_path, 'r', encoding='utf-8') as f:
    test_cases = json.load(f)

print(f"테스트 케이스 수: {len(test_cases)}")
print(f"\n샘플 테스트 케이스:")
print(json.dumps(test_cases[0], indent=2, ensure_ascii=False))

In [ ]:
# 테스트 케이스 분포 확인
from collections import Counter

tool_dist = Counter(tc['expected_tool'] for tc in test_cases)
complexity_dist = Counter(tc['complexity'] for tc in test_cases)

print("Tool 분포:")
for tool, count in tool_dist.most_common():
    print(f"  {tool}: {count}")

print(f"\n복잡도 분포:")
for complexity, count in complexity_dist.most_common():
    print(f"  {complexity}: {count}")

## 4. Tool Calling 평가

In [ ]:
def generate_response(model, tokenizer, user_query, max_new_tokens=512):
    """모델로부터 응답 생성"""
    
    # Tool 정의를 포함한 시스템 프롬프트
    tools_json = json.dumps(
        [{"name": name, **tool._asdict()} 
         for name, tool in AVAILABLE_TOOLS.items()],
        ensure_ascii=False
    )
    
    system_message = f"{SYSTEM_PROMPT}\n\n사용 가능한 도구:\n{tools_json}"
    
    # Qwen2.5 채팅 형식
    messages = [
        {"role": "system", "content": system_message},
        {"role": "user", "content": user_query}
    ]
    
    # 토큰화
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    
    # 생성
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.1,
            do_sample=True,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    
    # 응답 추출
    response = tokenizer.decode(
        outputs[0][inputs['input_ids'].shape[1]:],
        skip_special_tokens=True
    )
    
    return response

print("응답 생성 함수 정의 완료")

In [ ]:
# 단일 테스트 케이스로 확인
test_query = test_cases[0]['query']
print(f"테스트 쿼리: {test_query}")
print("\n응답 생성 중...")

response = generate_response(model, tokenizer, test_query)
print(f"\n모델 응답:\n{response}")

# Tool Call 추출
extracted = extract_tool_call(response)
print(f"\n추출된 Tool Call: {extracted}")

In [ ]:
# 전체 테스트 케이스 평가
print(f"전체 {len(test_cases)}개 테스트 케이스 평가 시작...")
print("(약 10-20분 소요될 수 있습니다)\n")

results = []

for i, tc in enumerate(tqdm(test_cases, desc="평가 진행")):
    try:
        # 응답 생성
        response = generate_response(model, tokenizer, tc['query'])
        
        # Tool Call 추출
        predicted = extract_tool_call(response)
        
        # 예상 결과와 비교
        expected = {
            "name": tc['expected_tool'],
            "arguments": tc['expected_arguments']
        }
        
        is_correct, error_type = compare_tool_calls(predicted, expected)
        
        results.append({
            "id": tc['id'],
            "query": tc['query'],
            "expected_tool": tc['expected_tool'],
            "expected_arguments": tc['expected_arguments'],
            "predicted": predicted,
            "response": response,
            "is_correct": is_correct,
            "error_type": error_type,
            "complexity": tc['complexity']
        })
        
    except Exception as e:
        results.append({
            "id": tc['id'],
            "query": tc['query'],
            "expected_tool": tc['expected_tool'],
            "expected_arguments": tc['expected_arguments'],
            "predicted": None,
            "response": str(e),
            "is_correct": False,
            "error_type": "generation_error",
            "complexity": tc['complexity']
        })

print(f"\n평가 완료! 총 {len(results)}개 케이스")

In [ ]:
# 전체 정확도 계산
correct_count = sum(1 for r in results if r['is_correct'])
total_count = len(results)
accuracy = correct_count / total_count * 100

print("=" * 60)
print(f"전체 정확도: {accuracy:.2f}% ({correct_count}/{total_count})")
print("=" * 60)

In [ ]:
# Tool별 정확도
tool_results = {}
for r in results:
    tool = r['expected_tool']
    if tool not in tool_results:
        tool_results[tool] = {'correct': 0, 'total': 0}
    tool_results[tool]['total'] += 1
    if r['is_correct']:
        tool_results[tool]['correct'] += 1

print("\nTool별 정확도:")
for tool, stats in tool_results.items():
    acc = stats['correct'] / stats['total'] * 100
    print(f"  {tool}: {acc:.1f}% ({stats['correct']}/{stats['total']})")

In [ ]:
# 복잡도별 정확도
complexity_results = {}
for r in results:
    comp = r['complexity']
    if comp not in complexity_results:
        complexity_results[comp] = {'correct': 0, 'total': 0}
    complexity_results[comp]['total'] += 1
    if r['is_correct']:
        complexity_results[comp]['correct'] += 1

print("\n복잡도별 정확도:")
for comp, stats in complexity_results.items():
    acc = stats['correct'] / stats['total'] * 100
    print(f"  {comp}: {acc:.1f}% ({stats['correct']}/{stats['total']})")

## 5. 에러 분석

In [ ]:
# 에러 유형 분포
error_types = Counter(r['error_type'] for r in results if not r['is_correct'])

print("에러 유형 분포:")
for error_type, count in error_types.most_common():
    print(f"  {error_type}: {count}")

In [ ]:
# 에러 케이스 상세 분석
error_cases = [r for r in results if not r['is_correct']]

print(f"총 에러 케이스: {len(error_cases)}개\n")

# 첫 5개 에러 케이스 출력
for i, case in enumerate(error_cases[:5]):
    print(f"--- 에러 케이스 {i+1} ---")
    print(f"ID: {case['id']}")
    print(f"Query: {case['query']}")
    print(f"Expected: {case['expected_tool']}({case['expected_arguments']})")
    print(f"Predicted: {case['predicted']}")
    print(f"Error Type: {case['error_type']}")
    print()

In [ ]:
# 에러 유형별 시각화
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Tool별 정확도 바 차트
tools = list(tool_results.keys())
accuracies = [tool_results[t]['correct'] / tool_results[t]['total'] * 100 for t in tools]

axes[0].barh(tools, accuracies, color='steelblue')
axes[0].set_xlabel('Accuracy (%)')
axes[0].set_title('Accuracy by Tool')
axes[0].set_xlim(0, 100)
for i, v in enumerate(accuracies):
    axes[0].text(v + 1, i, f'{v:.1f}%', va='center')

# 에러 유형 파이 차트
if error_types:
    axes[1].pie(
        error_types.values(),
        labels=error_types.keys(),
        autopct='%1.1f%%',
        startangle=90
    )
    axes[1].set_title('Error Type Distribution')
else:
    axes[1].text(0.5, 0.5, 'No Errors!', ha='center', va='center', fontsize=20)
    axes[1].set_title('Error Type Distribution')

plt.suptitle(f"{config['experiment']['name']} Evaluation Results", fontsize=14)
plt.tight_layout()

output_dir = f"../{config['misc']['output_dir']}"
plt.savefig(os.path.join(output_dir, 'evaluation_results.png'), dpi=300)
plt.show()

## 6. 결과 저장

In [ ]:
# 평가 결과 요약
evaluation_summary = {
    "experiment": config['experiment']['name'],
    "total_test_cases": total_count,
    "correct_count": correct_count,
    "overall_accuracy": accuracy,
    "accuracy_by_tool": {
        tool: {
            "accuracy": stats['correct'] / stats['total'] * 100,
            "correct": stats['correct'],
            "total": stats['total']
        }
        for tool, stats in tool_results.items()
    },
    "accuracy_by_complexity": {
        comp: {
            "accuracy": stats['correct'] / stats['total'] * 100,
            "correct": stats['correct'],
            "total": stats['total']
        }
        for comp, stats in complexity_results.items()
    },
    "error_distribution": dict(error_types)
}

print(json.dumps(evaluation_summary, indent=2, ensure_ascii=False))

In [ ]:
# 결과 저장
output_dir = f"../{config['misc']['output_dir']}"

# 평가 요약 저장
summary_path = os.path.join(output_dir, 'evaluation_summary.json')
with open(summary_path, 'w', encoding='utf-8') as f:
    json.dump(evaluation_summary, f, indent=2, ensure_ascii=False)
print(f"평가 요약 저장: {summary_path}")

# 상세 결과 저장
detailed_path = os.path.join(output_dir, 'evaluation_detailed.json')
with open(detailed_path, 'w', encoding='utf-8') as f:
    json.dump(results, f, indent=2, ensure_ascii=False)
print(f"상세 결과 저장: {detailed_path}")

In [ ]:
# 최종 결과 출력
print("\n" + "=" * 60)
print(f"평가 완료: {config['experiment']['name']}")
print("=" * 60)
print(f"\n📊 전체 정확도: {accuracy:.2f}%")
print(f"\n📁 저장된 파일:")
print(f"  - 평가 요약: {summary_path}")
print(f"  - 상세 결과: {detailed_path}")
print(f"  - 시각화: {output_dir}/evaluation_results.png")

## 다음 단계

평가가 완료되었습니다. 다음 노트북에서 실험 결과를 종합 분석합니다:
- `04_analysis.ipynb`